# Tahap 1. Setup & Load Dataset

## 1.1 Requirements & Setup GPU (MPS)

In [2]:
#pip install torch transformers datasets evaluate seqeval pandas ipywidgets

In [3]:
import torch

# Cek ketersediaan MPS untuk Apple Silicon
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Menggunakan MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print("MPS tidak tersedia, menggunakan CPU")


Menggunakan MPS (Apple Silicon GPU)


## 1.2. Loading NERGrit Dataset

Pada terminal, saya melakukan ini utk mendownload raw data filenya dan memasukkan ke folder data_nergri. Datasetnya dari repositori Cahya Wirawan (nergrit-corpus_20190726_corrected.tgz) berisi arsip data mentah NERGrit versi lengkap 18 entitas


**Membuat Buat folder data lokal**

mkdir -p data_nergrit18

**Unduh file arsip .tgz**

curl -L -o data_nergrit18/nergrit.tgz https://github.com/cahya-wirawan/indonesian-language-models/raw/master/data/nergrit-corpus_20190726_corrected.tgz

**Ekstrak arsip .tgz**

tar -xvf data_nergrit18/nergrit.tgz -C data_nergrit18/

In [4]:
# Mengecek  Struktur File Hasil Ekstraksi
import os

print("File/Folder hasil ekstraksi:")
for root, dirs, files in os.walk("data_nergrit18"):
    for file in files:
        if not file.startswith('.'):
            path = os.path.join(root, file)
            print(f"- {path} ({os.path.getsize(path)} bytes)")

File/Folder hasil ekstraksi:
- data_nergrit18/nergrit.tgz (14987801 bytes)
- data_nergrit18/nergrit-corpus/LICENSE (1472 bytes)
- data_nergrit18/nergrit-corpus/Readme.md (2868 bytes)
- data_nergrit18/nergrit-corpus/README.md.pdf (67552 bytes)
- data_nergrit18/nergrit-corpus/ner/ner_new.py (1588 bytes)
- data_nergrit18/nergrit-corpus/ner/make_model.py (1611 bytes)
- data_nergrit18/nergrit-corpus/ner/anagojsono.py (4511 bytes)
- data_nergrit18/nergrit-corpus/ner/tag.py (1523 bytes)
- data_nergrit18/nergrit-corpus/ner/model/nerprepro.pkl (676223 bytes)
- data_nergrit18/nergrit-corpus/ner/model/nerparams.json (14548 bytes)
- data_nergrit18/nergrit-corpus/ner/model/nerweight.h5 (5050500 bytes)
- data_nergrit18/nergrit-corpus/ner/data/log.csv (471 bytes)
- data_nergrit18/nergrit-corpus/ner/data/train.txt (2979492 bytes)
- data_nergrit18/nergrit-corpus/ner/data/valid.txt (594359 bytes)
- data_nergrit18/nergrit-corpus/ner/data/labels-map.csv (1405 bytes)
- data_nergrit18/nergrit-corpus/ner/dat

**Membuat Rasional Fungsi Custom Parser `parse_nergrit_file`**

Penggunaan fungsi parser kustom `parse_nergrit_file` dilakukan berdasarkan pertimbangan teknis berikut:

1. **Format Struktur CoNLL/TSV Mentah:** Berkas data NERGrit (`train.txt`, `valid.txt`, `test.txt`) tersimpan dalam format CoNLL/TSV. Setiap baris berisi pasangan `[token] [tag]` yang dipisahkan oleh spasi/tab, sedangkan baris kosong (`""`) berperan sebagai penanda batas antar-kalimat.

2. **Standardisasi Format Hugging Face:** Fungsi parser membaca berkas baris demi baris untuk mengelompokkan token dan tag per kalimat ke dalam *list*, kemudian mengonversinya menjadi kamus `{"tokens": [...], "ner_tags": [...]}`. Struktur ini merupakan format wajib agar data dapat diubah menjadi objek `Dataset` dan `DatasetDict` Hugging Face.

3. **Presisi Jalur Data (Isolasi Task NER):** Memuat berkas secara langsung melalui *path* eksplisit (`.../ner/data/`) menjamin bahwa data yang diproses secara presisi adalah dataset Named Entity Recognition (NER), sehingga menghindari risiko salah muat direktori (seperti direktori `sentiment` atau `statement`).

In [5]:
from datasets import Dataset, DatasetDict

# Membuat fungsi parser format CoNLL/TSV
def parse_nergrit_file(file_path):
    all_tokens = []
    all_tags = []
    
    current_tokens = []
    current_tags = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            
            if not line:
                if current_tokens:
                    all_tokens.append(current_tokens)
                    all_tags.append(current_tags)
                    current_tokens = []
                    current_tags = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    current_tokens.append(parts[0])
                    current_tags.append(parts[-1])
                    
        if current_tokens:
            all_tokens.append(current_tokens)
            all_tags.append(current_tags)
            
    return {"tokens": all_tokens, "ner_tags": all_tags}

# Path sesuai struktur folder ke dataset
train_path = "data_nergrit18/nergrit-corpus/ner/data/train.txt"
valid_path = "data_nergrit18/nergrit-corpus/ner/data/valid.txt"
test_path  = "data_nergrit18/nergrit-corpus/ner/data/test.txt"

# Parsing & Konversi ke DatasetDict
train_data = parse_nergrit_file(train_path)
valid_data = parse_nergrit_file(valid_path)
test_data  = parse_nergrit_file(test_path)

dataset_full = DatasetDict({
    'train': Dataset.from_dict(train_data),
    'validation': Dataset.from_dict(valid_data),
    'test': Dataset.from_dict(test_data)
})

# Verifikasi Hasil & Keberadaan Entitas DAT
print("=== Ringkasan DatasetDict (NER 18 Entitas) ===")
print(dataset_full)

all_tags_full = [tag for sentence in dataset_full['train']['ner_tags'] for tag in sentence]
unique_tags_full = sorted(list(set(all_tags_full)))

print(f"\nTotal Tag Unik : {len(unique_tags_full)}")
print("Daftar Tag Unik:")
print(unique_tags_full)

has_dat = any('DAT' in tag for tag in unique_tags_full)

# Untuk mengecek entitas DAT (Date), karena sebelumnya tidak ada di dataset yg saya pakai meskipun masih data NERGris
print(f"\nApakah entitas DAT (Date) ditemukan? -> {has_dat}")

=== Ringkasan DatasetDict (NER 18 Entitas) ===
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 12514
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2520
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2397
    })
})

Total Tag Unik : 126
Daftar Tag Unik:
['-', '0', '6', '8', '9', 'B', 'B-BAT', 'B-BGPE', 'B-CRD', 'B-CRR', 'B-CTY', 'B-DAT', 'B-ER', 'B-EVE', 'B-EVT', 'B-EVT`', 'B-FA', 'B-FAC', 'B-GPE', 'B-L0C', 'B-LAN', 'B-LAW', 'B-LOC', 'B-M0N', 'B-MON', 'B-N0R', 'B-NOE', 'B-NOR', 'B-NPR', 'B-O', 'B-ODR', 'B-OR', 'B-ORD', 'B-ORG', 'B-PE', 'B-PER', 'B-PER.', 'B-PGE', 'B-PR', 'B-PRC', 'B-PRD', 'B-PRG', 'B-PROD', 'B-QT', 'B-QTY', 'B-QYT', 'B-REG', 'B-TIM', 'B-WOA', 'B-WTY', 'B=CRD', 'B=GPE', 'B=PER', 'B=PRD', 'BGPE', 'BPER', 'BPRD', 'BWOA', 'CRD', 'DAT', 'GPE', 'I', 'I-0RG', 'I-AC', 'I-BTN', 'I-CRD', 'I-DAI', 'I-DAR', 'I-DAT', 'I-DaT', 'I-EVE', 'I-EVT', 'I-FA',

**Berdasarkan hasil eksekusi pemuatan dan verifikasi data:**

1. **Volume Data yang Melimpah:**
   - **Train Set:** 12.514 kalimat
   - **Validation Set:** 2.520 kalimat
   - **Test Set:** 2.397 kalimat  
   Total data sebanyak **17.431 kalimat** (~300 ribu+ token) menyediakan sampel pelatihan yang sangat ideal untuk proses *fine-tuning* model *transformer* `bert-base-multilingual-cased`.

2. **Identifikasi Tag Mentah & Noise Anotasi (126 Tag Unik):**
   - Terdeteksi sebanyak **126 tag unik** pada korpus mentah.
   - Banyaknya variasi tag ini disebabkan oleh skema bawaan NERGrit yang mencakup 18 jenis entitas, ditambah adanya *typo* atau variasi anotasi manual (seperti `B-L0C`, `I-DaT`, `B=PER`, `O‚Äé`).
   - Temuan ini memberikan justifikasi kuat untuk melakukan penyaringan (*filtering*) dan pemetaan (*mapping*) ketat pada **Tahap 3** menjadi **9 tag BIO resmi** (4 entitas target + 1 tag `O`).

3. **Konfirmasi Keberadaan Entitas Tanggal (`DAT = True`):**
   - Pengujian keberadaan entitas `DAT` menghasilkan nilai **`True`** (terbukti dengan hadirnya tag `B-DAT` dan `I-DAT`).
   - Temuan ini memvalidasi bahwa dataset NERGrit yang dimuat telah memenuhi 100% kriteria spesifikasi tugas, khususnya keberadaan entitas **Date (Tanggal)** yang tidak dimiliki oleh alternatif korpus NER Indonesia lainnya.

# Tahap 2: Eksplorasi Data

## 2.1 Analisis Distribusi Tag & Class Imbalance

In [6]:
# Distribusi Tag & Analisis Imbalance
from collections import Counter
import pandas as pd
import numpy as np

# Meratakan seluruh tag di train split
all_train_tags = [tag for sentence in dataset_full['train']['ner_tags'] for tag in sentence]
tag_counts = Counter(all_train_tags)

# Buat DataFrame distribusi tag
df_tags = pd.DataFrame(tag_counts.items(), columns=['Tag', 'Count']).sort_values(by='Count', ascending=False)
df_tags['Percentage (%)'] = round((df_tags['Count'] / len(all_train_tags)) * 100, 2)

print(f"Total Token di Train Set : {len(all_train_tags):,}")
print(f"Total Tag Unik           : {len(df_tags)}")
print("\n=== Top 20 Tag Terbanyak ===")
print(df_tags.head(20).to_string(index=False))

Total Token di Train Set : 309,202
Total Tag Unik           : 126

=== Top 20 Tag Terbanyak ===
  Tag  Count  Percentage (%)
    O 213728           69.12
I-DAT   9892            3.20
B-PER   6715            2.17
B-GPE   6447            2.09
I-NOR   5540            1.79
I-ORG   5418            1.75
I-LOC   4525            1.46
I-PER   4359            1.41
I-PRD   4175            1.35
B-ORG   4171            1.35
B-PRD   4135            1.34
B-NOR   3889            1.26
B-DAT   3585            1.16
B-CRD   3424            1.11
I-MON   3249            1.05
B-LOC   2841            0.92
I-QTY   2441            0.79
I-EVT   2384            0.77
I-TIM   2360            0.76
I-GPE   2315            0.75


Berdasarkan hasil eksplorasi distribusi label pada *train set* (total **309.202 token**):

1. **Dominasi Class 'O' (Outside):** Tag `O` mendominasi sebesar **69,12%** (213.728 token). Ketimpangan kelas (*class imbalance*) ini merupakan karakteristik alami pada *task* Named Entity Recognition (NER) karena mayoritas kata dalam teks bukan merupakan bagian dari entitas khusus.

2. **Ketersediaan 4 Entitas Target:** Seluruh entitas yang dibutuhkan dalam tugas terwakili secara memadai di jajaran tag teratas:
   - **DAT (Date):** `I-DAT` (3,20%) dan `B-DAT` (1,16%).
   - **PER (Person):** `B-PER` (2,17%) dan `I-PER` (1,41%).
   - **ORG (Organization):** `I-ORG` (1,75%) dan `B-ORG` (1,35%).
   - **LOC & GPE (Location):** `B-GPE` (2,09%), `I-LOC` (1,46%), `B-LOC` (0,92%), dan `I-GPE` (0,75%).

3. **Temuan Noise pada Tag Asli:** Terdeteksi **126 tag unik** pada dataset mentah. Tingginya variasi ini disebabkan oleh *typo* atau ketidakkonsistenan anotasi manual (seperti `B-L0C`, `I-DaT`, `B=PER`). Temuan ini memperkuat keputusan desain pada **Tahap 3** untuk melakukan *filtering* dan pemetaan ketat menjadi 9 label resmi (8 tag BIO untuk 4 entitas + 1 tag 'O').

## 2.2 Analisis Panjang Kalimat (Sequence Length)

In [ ]:
# Mengnalisis Panjang Kalimat (Panjang Sequence)
# Menghitung jumlah token per kalimat
sentence_lengths = [len(tokens) for tokens in dataset_full['train']['tokens']]

print("=== Statistik Panjang Kalimat (Token Count) ===")
print(f"Minimum    : {np.min(sentence_lengths)} token")
print(f"Rata-rata  : {np.mean(sentence_lengths):.2f} token")
print(f"Maksimum   : {np.max(sentence_lengths)} token")
print(f"Percentile 95% : {np.percentile(sentence_lengths, 95):.0f} token")
print(f"Percentile 99% : {np.percentile(sentence_lengths, 99):.0f} token")

=== Statistik Panjang Kalimat (Token Count) ===
Minimum    : 1 token
Rata-rata  : 24.71 token
Maksimum   : 467 token
Percentile 95% : 50 token
Percentile 99% : 83 token


Berdasarkan statistik panjang kalimat pada *train set*:

1. **Rata-Rata & Distribusi Utama:** Rata-rata panjang kalimat berada di angka **24,71 token**, di mana **95% data** memiliki panjang $\le$ 50 token dan **99% data** memiliki panjang $\le$ 83 token.
2. **Keputusan Desain `max_length`:** Karena 99% kalimat berada di bawah 83 token, penentuan nilai `max_length = 128` saat tokenisasi BERT merupakan pilihan yang sangat optimal. 
3. **Efisiensi Komputasi:** Batas 128 token memastikan seluruh informasi konteks pada >99% kalimat dapat ditangkap secara utuh tanpa terpotong (*truncation*), sekaligus menghemat memori GPU (MPS) serta mempercepat proses *fine-tuning*.

## 2.3 Inspeksi Anotasi Entitas Target & Skema BIO

In [7]:
# 3. Inspeksi Sampel Berentitas Tanggal (DAT)
print("=== Sampel Kalimat Berentitas Tanggal (DAT) ===")
dat_found = 0

for i, sample in enumerate(dataset_full['train']):
    if any('DAT' in t for t in sample['ner_tags']):
        print(f"\n[Kalimat Indeks ke-{i}]")
        for token, tag in zip(sample['tokens'], sample['ner_tags']):
            if 'DAT' in tag or tag in ['B-PER', 'B-ORG', 'B-LOC', 'B-GPE']:
                print(f"  {token:20} -> {tag}")
        
        dat_found += 1
        if dat_found >= 3:  # Tampilkan 3 contoh
            break

=== Sampel Kalimat Berentitas Tanggal (DAT) ===

[Kalimat Indeks ke-0]
  Indonesia            -> B-GPE
  Amerika              -> B-GPE
  Jakarta              -> B-GPE
  Selasa               -> B-DAT
  15                   -> I-DAT
  8                    -> I-DAT
  2018                 -> I-DAT
  Jakarta              -> B-LOC
  JICT                 -> B-LOC
  Selasa               -> B-DAT
  15                   -> I-DAT
  5                    -> I-DAT
  2018                 -> I-DAT
  West                 -> B-LOC
  LA                   -> B-GPE
  Oakland              -> B-GPE
  Amerika              -> B-GPE
  Presiden             -> B-PER
  Amerika              -> B-GPE
  Indonesia            -> B-GPE
  Indo                 -> B-LOC
  Indonesia            -> B-GPE
  kawasan              -> B-LOC
  Presiden             -> B-PER
  Budi                 -> B-PER
  Airlangga            -> B-PER
  Enggartiasto         -> B-PER
  Rini                 -> B-PER
  IPC                  -> B-ORG
 

Berdasarkan hasil inspeksi sampel data berentitas tanggal (`DAT`):

1. **Konsistensi Skema BIO pada Tanggal (`DAT`):** Anotasi tanggal berhasil mencakup berbagai variasi penulisan tanggal dalam bahasa Indonesia, baik format teks ("Selasa 15 8 2018"), format angka dengan simbol pemisah ("Rabu , 16 / 05 / 2018"), maupun nama hari/bulan ("13 Mei", "Senin"). Token awal konsisten diawali `B-DAT` dan token lanjutannya (termasuk simbol `,` dan `/`) diberi tag `I-DAT`.
2. **Konfirmasi Keberadaan 4 Entitas Target:** Sampel terbukti kaya akan entitas yang sesuai dengan spesifikasi tugas: `PER` ("Budi", "Airlangga"), `ORG` ("CNN", "IPC"), `DAT` ("13 Mei"), serta entitas lokasi `GPE` dan `LOC`.
3. **Justifikasi Penggabungan `GPE` & `LOC`:** Pada sampel terlihat entitas geopolitik seperti negara/kota ("Indonesia", "Jakarta", "Surabaya") terlabeli `GPE`, sedangkan area geografis/fasilitas ("JICT", "kawasan") terlabeli `LOC`. Temuan ini semakin memperkuat keputusan desain pada **Tahap 3** untuk menggabungkan `GPE` dan `LOC` menjadi satu entitas payung `LOC`.

# Tahap 3: Preprocessing & Filtering

## 3.1 Menentukan Label ID & Skema Mapping

In [8]:
# List 9 tag resmi yang diinginkan
target_tags = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-DAT", "I-DAT"
]

# Kamus konversi Tag <-> ID
label2id = {tag: i for i, tag in enumerate(target_tags)}
id2label = {i: tag for i, tag in enumerate(target_tags)}

print("Pemetaan Label ke ID (9 Class):")
print(label2id)

Pemetaan Label ke ID (9 Class):
{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-DAT': 7, 'I-DAT': 8}


Berdasarkan keputusan desain pada **Tahap 3**, seluruh 126 tag mentah dari NERGrit telah disaring dan dipetakan secara eksklusif menjadi **9 kelas resmi** dalam skema BIO:

1. **Cakupan 4 Entitas Utama:** Pemetaan mencakup entitas `PER` (Orang), `ORG` (Organisasi), `LOC` (Lokasi, termasuk penggabungan `GPE`), dan `DAT` (Tanggal), serta tag `O` (Outside).
2. **Skema Tagging BIO:** Setiap entitas menggunakan prefix `B-` (Begin) untuk menandai awal munculnya entitas dan `I-` (Inside) untuk menandai token lanjutan dari entitas multi-kata.
3. **Penyelarasan ID Numerik:** Pemetaan string ke ID integer (`0` sampai `8`) disiapkan untuk memenuhi input standar arsitektur `BertForTokenClassification` pada PyTorch/Hugging Face.

### 3.2 Membuat Function Tag Cleaning & Mapping

Sebelum menjalankan eksekusi pemetaan pada dataset, fungsi `map_tag_to_target(tag)` dibuat tepat setelah penentuan `label2id` (3.1) sebagai **jembatan transformasi data**. Berikut alasan teknis dan tujuannya:

1. **Penyederhanaan dari 126 Tag ke 9 Target Label:**
   Dataset mentah NERGrit memuat 18 entitas dengan total 126 variasi tag (termasuk *noise* dan *typo*). Fungsi ini bertugas memangkas seluruh variasi tersebut agar persis sesuai dengan spesifikasi tugas yang hanya membutuhkan **4 entitas utama** (`PER`, `ORG`, `LOC`, `DAT`) dalam skema BIO + `O`.

2. **Penggabungan Entitas Geopolitik (`GPE`) ke Lokasi (`LOC`):**
   Di dataset NERGrit, nama kota/negara dilabeli sebagai `GPE` (*Geopolitical Entity*) sedangkan bentuk fisik seperti sungai/gunung dilabeli `LOC`. Untuk memenuhi instruksi tugas "Lokasi", fungsi ini secara eksplisit mengabungkan tag `GPE` ke dalam `LOC`.

3. **Filtering Entitas Non-Target Menjadi Class `'O'`:**
   Entitas lain yang tidak diminta oleh instruksi tugas (seperti `MON`/Uang, `CRD`/Angka, `PRD`/Produk, `NOR`/Demografi) serta tag yang mengalami *typo* (seperti `B-L0C` atau `B=PER`) secara otomatis dinormalisasi menjadi tag `'O'` (*Outside*).

4. **Koneksi Langsung ke `label2id`:**
   Dibuatnya fungsi ini setelah tahap 3.1 memastikan bahwa seluruh output string yang dihasilkan oleh fungsi pemetaan dijamin 100% cocok dengan *key* yang ada pada kamus `label2id`, sehingga tidak akan memicu error `KeyError` saat mengonversi tag teks menjadi ID numerik untuk model BERT.

In [9]:
def map_tag_to_target(raw_tag):
    # Bersihkan string dari noise karakter jika ada
    raw_tag = str(raw_tag).strip().upper()
    
    if raw_tag == "O" or not ("B-" in raw_tag or "I-" in raw_tag or "B=" in raw_tag):
        return "O"
    
    # Ambil prefix (B/I) dan root entitas
    prefix = "B-" if raw_tag.startswith("B") else "I-"
    
    # Deteksi entitas utama
    if "PER" in raw_tag:
        return f"{prefix}PER"
    elif "ORG" in raw_tag:
        return f"{prefix}ORG"
    elif "DAT" in raw_tag:
        return f"{prefix}DAT"
    elif "LOC" in raw_tag or "GPE" in raw_tag:
        return f"{prefix}LOC"
    else:
        return "O"

# Terapkan mapping ke seluruh dataset
def convert_dataset_labels(example):
    example["cleaned_tags"] = [map_tag_to_target(t) for t in example["ner_tags"]]
    example["label_ids"] = [label2id[t] for t in example["cleaned_tags"]]
    return example

# Mapping ke seluruh split dataset
dataset_mapped = dataset_full.map(convert_dataset_labels)

Map:   0%|          | 0/12514 [00:00<?, ? examples/s]

Map:   0%|          | 0/2520 [00:00<?, ? examples/s]

Map:   0%|          | 0/2397 [00:00<?, ? examples/s]

Berdasarkan hasil eksekusi transformasi data pada ketiga *split*:

1. **Keberhasilan Pemrosesan Data:** Fungsi pemetaan label berhasil dijalankan 100% tanpa hambatan pada seluruh *split* dataset, mencakup **12.514 kalimat** (*train*), **2.520 kalimat** (*validation*), dan **2.397 kalimat** (*test*).

2. **Penambahan Fitur Baru:** Setiap sampel dalam dataset kini telah memiliki dua kolom baru:
   - `cleaned_tags`: Tag teks yang telah dinormalisasi menjadi 9 tag BIO target.
   - `label_ids`: Representasi ID numerik integer (`0`–`8`) yang siap diproses oleh PyTorch.
   
3. **Kesiapan Subword Alignment:** Seluruh dataset kini memiliki struktur label yang bersih dari *noise* dan siap diumpankan ke tahap *subword tokenization & label alignment* dengan tokenizer `bert-base-multilingual-cased`.

### 3.2 Tokenisasi & Subword Label Alignment
Proses ini menyelaraskan label NER dengan hasil tokenisasi *WordPiece* milik BERT (`bert-base-multilingual-cased`):

1. **Pemecahan Subword:** Tokenizer memecah kata menjadi beberapa *subword* (contoh: *"Airlangga"* $\rightarrow$ `['Air', '##lang', '##ga']`). Hanya *subword* pertama yang mempertahankan label asli kata tersebut.
2. **Masking Label (`-100`):** Token spesial (`[CLS]`, `[SEP]`) serta *subword* pecahan (`##...`) diberi ID label **`-100`**.
3. **Pengabaian Loss:** Label ID `-100` secara otomatis diabaikan oleh fungsi *CrossEntropyLoss* PyTorch selama proses pelatihan agar perhitungan *loss* tetap akurat.

In [10]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples, max_length=128):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=max_length,
        is_split_into_words=True
    )

    labels = []
    for i, label_list in enumerate(examples["label_ids"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Peta token ke indeks kata asal
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Token spesial ([CLS], [SEP]) diberi label -100
            if word_idx is None:
                label_ids.append(-100)
            # Token pertama dari suatu kata mendapat label aslinya
            elif word_idx != previous_word_idx:
                label_ids.append(label_list[word_idx])
            # Subword berikutnya dari kata yang sama diberi label -100
            else:
                label_ids.append(-100)
            
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Eksekusi tokenisasi ke seluruh dataset
tokenized_datasets = dataset_mapped.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset_mapped["train"].column_names
)

Map:   0%|          | 0/12514 [00:00<?, ? examples/s]

Map:   0%|          | 0/2520 [00:00<?, ? examples/s]

Map:   0%|          | 0/2397 [00:00<?, ? examples/s]

Berdasarkan hasil eksekusi pemetaan fungsi `tokenize_and_align_labels` pada seluruh *split* dataset:

1. **Keberhasilan Pemrosesan Data:** Fungsi tokenisasi dan penyelarasan label berhasil dijalankan 100% tanpa kendala pada ketiga *split* data (**12.514 sampel train**, **2.520 sampel validation**, dan **2.397 sampel test**).

2. **Kelengkapan Fitur Input BERT:** Seluruh sampel teks kini telah dikonversi menjadi struktur tensor standar BERT, yang mencakup atribut `input_ids`, `attention_mask`, dan `labels` (dengan *subword masking* `-100`).

3. **Kesiapan Tahap Fine-Tuning:** Dataset telah terstandardisasi menggunakan batas `max_length = 128` dan siap diumpankan ke *Data Collator* serta *Trainer* pada **Tahap 4 (Fine-Tuning BERT)**.

## Kode Verifikasi Tahap 3 (Sanity Check)

In [11]:
# Ambil 1 sampel yang sudah ditokenisasi
sample_index = 0
tokens = tokenizer.convert_ids_to_tokens(tokenized_datasets["train"][sample_index]["input_ids"])
labels = tokenized_datasets["train"][sample_index]["labels"]

print("=== Hasil Tokenisasi & Alignment Subword BERT ===")
for tok, lbl in zip(tokens[:20], labels[:20]):
    tag_name = id2label[lbl] if lbl != -100 else "IGN (-100)"
    print(f"{tok:20} -> {lbl:4} ({tag_name})")

=== Hasil Tokenisasi & Alignment Subword BERT ===
[CLS]                -> -100 (IGN (-100))
Indonesia            ->    5 (B-LOC)
men                  ->    0 (O)
##ge                 -> -100 (IGN (-100))
##ks                 -> -100 (IGN (-100))
##por                -> -100 (IGN (-100))
produk               ->    0 (O)
industri             ->    0 (O)
skala                ->    0 (O)
besar                ->    0 (O)
ke                   ->    0 (O)
Amerika              ->    5 (B-LOC)
Serikat              ->    6 (I-LOC)
.                    ->    0 (O)
Ek                   ->    0 (O)
##spor               -> -100 (IGN (-100))
dilakukan            ->    0 (O)
melalui              ->    0 (O)
Pela                 ->    0 (O)
##buhan              -> -100 (IGN (-100))


Berdasarkan sampel hasil verifikasi *token alignment*:

1. **Masking Token Spesial:** Token awal `[CLS]` secara tepat diberi ID label **`-100`** sehingga diabaikan oleh kalkulasi *loss function*.
2. **Validasi Subword Alignment:** Kata-kata yang dipecah oleh *WordPiece Tokenizer* BERT berhasil diselaraskan dengan sempurna:
   - Kata *"mengekspor"* dipecah menjadi `['men', '##ge', '##ks', '##por']`. Subword pertama (`men`) mempertahankan label `0` (`O`), sedangkan pecahan berikutnya (`##ge`, `##ks`, `##por`) diberi label `-100`.
   - Pola yang sama terlihat pada kata *"Ekspor"* (`['Ek', '##spor']`) dan *"Pelabuhan"* (`['Pela', '##buhan']`).
3. **Presisi BIO Tagging pada Entitas Multi-Kata:** Entitas lokasi multi-kata *"Amerika Serikat"* secara tepat merepresentasikan struktur BIO: token *"Amerika"* sebagai `B-LOC` (ID `5`) dan token *"Serikat"* sebagai `I-LOC` (ID `6`).

**Kesimpulan Tahap 3:** *Pipeline data preprocessing* dan *label alignment* telah 100% valid. Seluruh dataset kini siap diumpankan ke **Tahap 4: Fine-Tuning Model**.

# Tahap 4: Fine-Tuning BERT

In [12]:
#%pip install -U accelerate seqeval

In [14]:
import numpy as np
import evaluate
from transformers import (
    AutoModelForTokenClassification, 
    DataCollatorForTokenClassification, 
    TrainingArguments, 
    Trainer
)

# 1. Load Metric Evaluasi seqeval (Standar Evaluasi NER)
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Filter out label -100 (token spesial & subwords) saat kalkulasi metrik
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 2. Inisialisasi Model bert-base-multilingual-cased
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(target_tags),
    id2label=id2label,
    label2id=label2id
)

# 3. Data Collator dengan Dynamic Padding
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 4. Konfigurasi TrainingArguments (Dioptimalkan untuk Apple MPS M2 Max)
training_args = TrainingArguments(
    output_dir="./results_bert_nergrit",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    report_to="none"  # Menonaktifkan logging ke wandb/external tools
)

# 5. Inisialisasi Trainer (Gunakan processing_class)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,  # Parameter yang disesuaikan
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 6. Eksekusi Training
print("=== Memulai Fine-Tuning BERT pada NERGrit ===")
train_result = trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

=== Memulai Fine-Tuning BERT pada NERGrit ===


/opt/anaconda3/envs/env_asignt_npl/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.099434,0.082402,0.840054,0.870423,0.854969,0.972705
2,0.068362,0.077482,0.867856,0.891080,0.879314,0.976043
3,0.045931,0.081553,0.876397,0.883803,0.880084,0.976910


...

/opt/anaconda3/envs/env_asignt_npl/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/envs/env_asignt_npl/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Proses *fine-tuning* `bert-base-multilingual-cased` selama 3 epoch (total **2.349 langkah**, ~21 menit) pada dataset NERGrit menunjukkan hasil konvergensi yang sangat solid:

1. **Peningkatan Performa Utama (F1-Score):**
   - **Epoch 1:** F1 = **85,50%** (Precision: 84,01%, Recall: 87,04%)
   - **Epoch 2:** F1 = **87,93%** (Precision: 86,79%, Recall: 89,11%)
   - **Epoch 3:** F1 = **88,01%** (Precision: 87,64%, Recall: 88,38%)
   - Model mencapai performa F1 puncak sebesar **88,01%** pada Epoch 3 dengan akurasi tingkat token sebesar **97,69%**.

2. **Dinamika Loss & Deteksi Overfitting:**
   - *Training Loss* terus turun secara konsisten dari `0.0994` (Epoch 1) hingga `0.0459` (Epoch 3), menandakan model berhasil mempelajari pola entitas dengan baik.
   - *Validation Loss* mencapai titik terendah (*sweet spot*) pada Epoch 2 (`0.0775`) dan mengalami kenaikan tipis pada Epoch 3 (`0.0815`). Hal ini mengindikasikan bahwa **3 epoch** adalah durasi pelatihan yang sangat pas sebelum model mengalami *overfitting*.

3. **Keseimbangan Precision & Recall:**
   - Tingkat *Precision* (87,64%) dan *Recall* (88,38%) sangat seimbang. Model tidak hanya mampu mendeteksi entitas target secara sensitif (*high recall*), tetapi juga akurat dan jarang memprediksi entitas palsu (*high precision*).

# Tahap 5 Evaluasi di Test Set

In [ ]:
# Evaluasi pada Test Set
print("=== Evaluasi Model pada Test Set ===")
test_results = trainer.evaluate(tokenized_datasets["test"])

print(f"Test Loss      : {test_results['eval_loss']:.4f}")
print(f"Test Precision : {test_results['eval_precision']:.4f}")
print(f"Test Recall    : {test_results['eval_recall']:.4f}")
print(f"Test F1-Score  : {test_results['eval_f1']:.4f}")
print(f"Test Accuracy  : {test_results['eval_accuracy']:.4f}")

# Hasilkan Prediksi Detail & Classification Report Per-Entitas
predictions, labels, _ = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

from seqeval.metrics import classification_report
print("\n=== Classification Report Per-Entitas (Test Set) ===")
print(classification_report(true_labels, true_predictions))

=== Evaluasi Model pada Test Set ===


/opt/anaconda3/envs/env_asignt_npl/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.045931,0.096496,3,0.866960,0.873733,0.870333,0.974910


Test Loss      : 0.0965
Test Precision : 0.8670
Test Recall    : 0.8737
Test F1-Score  : 0.8703
Test Accuracy  : 0.9749



=== Classification Report Per-Entitas (Test Set) ===
              precision    recall  f1-score   support

         DAT       0.93      0.94      0.94       788
         LOC       0.87      0.87      0.87      1644
         ORG       0.70      0.74      0.72       856
         PER       0.94      0.92      0.93      1448

   micro avg       0.87      0.87      0.87      4736
   macro avg       0.86      0.87      0.86      4736
weighted avg       0.87      0.87      0.87      4736



### 5.1 Evaluasi Per-Entitas pada Test Set

Berdasarkan hasil pengujian model `bert-base-multilingual-cased` pada *Test Set* (**2.397 kalimat** / **4.736 entitas**):

1. **Performa Generalisasi Sangat Baik:**
   - Model menghasilkan **Test F1-Score sebesar 87,03%** (Precision: 86,70%, Recall: 87,37%) dengan akurasi tingkat token sebesar **97,49%**.
   - Selisih F1-Score yang sangat tipis antara *Validation Set* (88,01%) dan *Test Set* (87,03%) membuktikan bahwa model tidak mengalami *overfitting* dan memiliki daya generalisasi yang sangat stabil pada data yang belum pernah dilihat sebelumnya.

2. **Analisis Performa Per-Entitas (Per-Class Performance):**
   - **DAT (Date) — F1: 94% (Precision: 0.93, Recall: 0.94):** 
     Merupakan entitas dengan performa tertinggi. Pola penulisan tanggal dalam bahasa Indonesia yang memiliki struktur spesifik (nama hari, bulan, dan format angka) memudahkan mBERT dalam mengenali batas entitas secara presisi. Temuan ini memvalidasi keputusan penggunaan dataset NERGrit.
   - **PER (Person) — F1: 93% (Precision: 0.94, Recall: 0.92):** 
     Menunjukkan performa yang sangat kuat. Model sangat presisi (*high precision*) dalam mengidentifikasi nama orang, dibantu oleh kapitalisasi kata serta konteks gelar/jabatan di sekitar nama.
   - **LOC (Location) — F1: 87% (Precision: 0.87, Recall: 0.87):** 
     Penggabungan `GPE` dan `LOC` terbukti efektif. Model berhasil mengenali entitas geopolitik (kota/negara) maupun fasilitas fisik/geografis dengan tingkat kestabilan yang seimbang antara *Precision* dan *Recall*.
   - **ORG (Organization) — F1: 72% (Precision: 0.70, Recall: 0.74):** 
     Merupakan entitas dengan performa paling rendah. Organisasi memiliki variasi penulisan yang sangat tinggi, sering kali tersusun dari kata-kata umum (*common words*), singkatan/akronim, atau nama tempat/geografis yang menyebabkan ambiguasi bagi model.

# Sampel Analisis Error (Tahap 6)

In [17]:
# Ekstraksi Prediksi Kalimat untuk Analisis Error
test_tokens = dataset_full['test']['tokens']
test_raw_tags = dataset_full['test']['ner_tags']

# Kumpulkan sampel untuk kualitatif analisis
correct_samples = []
error_samples = []
date_as_loc_samples = []

for idx, (tokens_sent, gold_tags_sent, pred_tags_sent) in enumerate(zip(test_tokens, true_labels, true_predictions)):
    # Petakan tag gold ke target 9 class
    mapped_gold = [map_tag_to_target(t) for t in gold_tags_sent]
    
    # Cek kesesuaian
    if mapped_gold == pred_tags_sent:
        if len(correct_samples) < 3 and any(t != 'O' for t in mapped_gold):
            correct_samples.append((tokens_sent, mapped_gold, pred_tags_sent))
    else:
        if len(error_samples) < 5:
            error_samples.append((tokens_sent, mapped_gold, pred_tags_sent))
            
        # Cek kasus spesifik: Tanggal diprediksi sebagai Lokasi (DAT -> LOC)
        for g, p in zip(mapped_gold, pred_tags_sent):
            if 'DAT' in g and 'LOC' in p:
                date_as_loc_samples.append((tokens_sent, mapped_gold, pred_tags_sent))
                break

print(f"Sampel Prediksi Benar Terkumpul : {len(correct_samples)}")
print(f"Sampel Prediksi Salah Terkumpul : {len(error_samples)}")
print(f"Kasus DAT keliru diprediksi LOC  : {len(date_as_loc_samples)}")

Sampel Prediksi Benar Terkumpul : 3
Sampel Prediksi Salah Terkumpul : 5
Kasus DAT keliru diprediksi LOC  : 0


In [18]:
# 1. Cetak Rincian Sampel Prediksi Benar
print("==================================================")
print("1. CONTOH PREDIKSI BENAR (CORRECT PREDICTIONS)")
print("==================================================")
for i, (toks, gold, pred) in enumerate(correct_samples, 1):
    print(f"\n--- Sampel Benar ke-{i} ---")
    for t, g, p in zip(toks, gold, pred):
        if g != 'O':  # Tampilkan hanya token entitas agar fokus
            print(f"{t:20} | Gold: {g:7} | Pred: {p:7} [MATCH]")

# 2. Cetak Rincian Sampel Prediksi Salah / Misklasifikasi
print("\n==================================================")
print("2. CONTOH PREDIKSI SALAH (ERROR SAMPLES)")
print("==================================================")
for i, (toks, gold, pred) in enumerate(error_samples, 1):
    print(f"\n--- Sampel Error ke-{i} ---")
    for t, g, p in zip(toks, gold, pred):
        if g != p:  # Tampilkan token yang mengalami kesalahan prediksi
            print(f"{t:20} | Gold: {g:7} | Pred: {p:7} [MISMATCH]")

# 3. Hitung Matriks Kebingungan (Confusion Matrix) Antar-Entitas Utama
from sklearn.metrics import confusion_matrix
import pandas as pd

# Flatten seluruh tag gold dan pred di test set (abaikan tag O agar fokus entitas)
flat_gold = [g for sublist in true_labels for g in sublist]
flat_pred = [p for sublist in true_predictions for p in sublist]

# Ambil entitas non-O
entity_labels = ["B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-DAT", "I-DAT", "O"]

cm = confusion_matrix(flat_gold, flat_pred, labels=entity_labels)
df_cm = pd.DataFrame(cm, index=[f"Actual {l}" for l in entity_labels], columns=[f"Pred {l}" for l in entity_labels])

print("\n==================================================")
print("3. MATRIKS KEBINGUNGAN (CONFUSION MATRIX) ENTITAS")
print("==================================================")
print(df_cm.to_string())

1. CONTOH PREDIKSI BENAR (CORRECT PREDICTIONS)

--- Sampel Benar ke-1 ---
Cianjur              | Gold: B-LOC   | Pred: B-LOC   [MATCH]
Utara                | Gold: I-LOC   | Pred: I-LOC   [MATCH]

--- Sampel Benar ke-2 ---
Waduk                | Gold: B-LOC   | Pred: B-LOC   [MATCH]
Jangari              | Gold: I-LOC   | Pred: I-LOC   [MATCH]
Gunung               | Gold: B-LOC   | Pred: B-LOC   [MATCH]
Padang               | Gold: I-LOC   | Pred: I-LOC   [MATCH]
Cibodas              | Gold: B-LOC   | Pred: B-LOC   [MATCH]

--- Sampel Benar ke-3 ---
Asep                 | Gold: B-PER   | Pred: B-PER   [MATCH]

2. CONTOH PREDIKSI SALAH (ERROR SAMPLES)

--- Sampel Error ke-1 ---
CIANJUR              | Gold: B-LOC   | Pred: B-ORG   [MISMATCH]
Kabupaten            | Gold: B-LOC   | Pred: O       [MISMATCH]
Cianjur              | Gold: I-LOC   | Pred: O       [MISMATCH]

--- Sampel Error ke-2 ---
Kabupaten            | Gold: B-LOC   | Pred: O       [MISMATCH]
Cianjur              | Gold: I-L

### 6.1 Analisis Kualitatif Prediksi & Pola Error (Error Analysis)

Berdasarkan contoh sampel prediksi dan tabel *Confusion Matrix* pada *Test Set*, berikut adalah analisis mendalam terhadap perilaku model `bert-base-multilingual-cased`:

#### 1. Analisis Prediksi Benar (Correct Predictions)
Model menunjukkan kapabilitas yang sangat kuat pada kondisi berikut:
* **Entitas Lokasi Spasial & Geografis:** Model berhasil mengenali nama lokasi yang diawali kata kunci geografis seperti *"Waduk Jangari"*, *"Gunung Padang"*, dan *"Cianjur Utara"*.
* **Nama Orang Lokal:** Nama khas Indonesia seperti *"Asep"* terdeteksi sebagai `B-PER` dengan presisi tinggi saat berada dalam struktur kalimat standar.

#### 2. Analisis Pola Error Spesifik (Misclassifications)
Teridentifikasi 3 pola kesalahan utama (*error patterns*) pada model:
* **Format Dateline Berita (All-Caps Text):** Kata seperti `"CIANJUR"` atau `"BANDUNG"` yang ditulis dengan huruf kapital di awal artikel berita sering dikira sebagai `B-ORG` atau luput dari `B-LOC`. Hal ini terjadi karena gaya penulisan *dateline* berita menggabungkan nama tempat dan agensi berita (contoh: *BANDUNG, PR - ...*).
* **Deskriptor Administratif Terlewat (`Kabupaten` / `Kota`):** Frasa *"Kabupaten Cianjur"* kadang terprediksi sebagai `O` (*False Negative*). Model terkadang gagal menangkap konteks bahwa kata deskriptor administratif merupakan bagian dari kesatuan entitas lokasi.
* **Ambiguasi Kata Umum & Noise Anotasi Dataset:** Kata *"Ide"* atau nama *"Asep"* yang muncul di konteks tertentu salah diprediksi sebagai entitas, atau justru merupakan akibat dari ketidakkonsistenan label mentah (*ground truth noise*) pada dataset.

#### 3. Evaluasi Hipotesis "Tanggal vs Lokasi" & Insight Confusion Matrix
* **Menjawab Pertanyaan Tugas (Apakah Tanggal Tertukar dengan Lokasi?):** **Tidak.** Berdasarkan *Confusion Matrix*, nilai `Actual B-DAT` atau `Actual I-DAT` yang diprediksi sebagai `LOC` adalah **0**. Tanggal (`DAT`) memiliki batas atribut sintaksis yang sangat kuat dalam bahasa Indonesia sehingga tidak pernah membingungkan model sebagai tempat.
* **Penyebab Rendahnya F1-Score Organisasi (`ORG` = 72%):** *Confusion Matrix* menunjukkan bahwa `B-ORG` dan `I-ORG` paling banyak meleset menjadi kelas `O` (156 dan 228 token) serta tertukar dengan `LOC` (22 token). Nama organisasi di Indonesia sering kali memakai nama tempat (contoh: *"Semen Padang"*, *"Bank DKI"*) atau tersusun dari kata-kata umum, sehingga mBERT kesulitan menentukan batas entitasnya.

---

### 6.2 Pembahasan Etika AI, Keamanan Data, dan Privasi Pengguna

Dalam konteks penerpan model NLP untuk analisis teks media massa:

1. **Keamanan Data & Privasi Pengguna:**
   - Dataset yang digunakan berasal dari korpus publik berita online. Tidak ditemukan informasi sensitif pribadi (*Personally Identifiable Information* / PII) seperti Nomor NIK, nomor rekening, alamat rumah pribadi, atau data medis.
   - Untuk penerapan pada data riil di masa depan, sistem pemrosesan wajib menambahkan tahapan *anonymization/pseudonymization* jika mengolah dokumen internal non-publik.
2. **Potensi Bias Penamaan (Nama Lokal vs Barat):**
   - Model pretrained `bert-base-multilingual-cased` dilatih dengan korpus multibahasa global. Terdapat risiko bias di mana nama-nama barat (contoh: *"John"*, *"Trump"*) memiliki *subword embeddings* yang lebih utuh, sedangkan nama lokal Indonesia sering terpecah menjadi beberapa *subwords*. Namun, berkat fine-tuning pada NERGrit, performa `PER` tetap mencapai F1 **93%**.
3. **Keseimbangan Topik Berita:**
   - Korpus NERGrit mencakup ranah berita umum (politik, ekonomi, olahraga, dan budaya). Evaluasi menunjukkan model tetap stabil karena entitas tanggal (`DAT`) dan lokasi (`LOC`) memiliki bentuk konsisten di berbagai domain topik.

---

### 7. Rekomendasi Improvisasi Model (Recommended Next Actions)

Untuk meningkatkan F1-Score model dari **87,03%** menuju **>90%**, berikut rekomendasi langkah teknis yang dapat dilakukan:

1. **Penggunaan Pretrained Model Spesifik Bahasa Indonesia:**
   - Mengganti backbone `bert-base-multilingual-cased` dengan model yang dilatih khusus pada korpus bahasa Indonesia, seperti **`indobenchmark/indobert-base-p1`** atau **`cased-indobert`**. Model lokal memiliki kosa kata (*vocabulary*) yang lebih relevan sehingga mengurangi pemecahan *subword* berlebih pada nama tempat dan orang lokal.
2. **Aturan Post-Processing & Pre-tokenization Normalisasi:**
   - Menambahkan aturan *post-processing* khusus untuk penanganan *dateline* berita (misal: memisahkan nama kota kapital di awal paragraf sebelum diumpankan ke model).
   - Menerapkan penanganan aturan khusus (*rule-based*) untuk frasa administratif seperti *"Kabupaten"* / *"Kota"* / *"Provinsi"* agar selalu terikat dengan kata setelahnya.
3. **Penanganan Class Imbalance pada Organisasi (`ORG`):**
   - Melakukan teknik *Focal Loss* atau penimbangan bobot kelas (*Class Weighting*) pada fungsi kerugian (*CrossEntropyLoss*) untuk memberikan perhatian lebih pada kelas `ORG` yang F1-score-nya masih tertinggal di angka 72%.
4. **Data Cleaning & Rule-Based Tag Consistency:**
   - Melakukan pembersihan dataset (*data sanitization*) pada label mentah untuk memperbaiki ketidakkonsistenan anotasi *ground truth* pada kata-kata kapitalisasi khusus.

# Menyimpan Model Hasil Training

In [19]:
# Simpan model dan tokenizer terbaik
output_dir = "./best_indonesian_ner_model"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model dan tokenizer berhasil disimpan di folder: {output_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model dan tokenizer berhasil disimpan di folder: ./best_indonesian_ner_model
